2025-07-22

Trying to pull apart how algorithm works

In [9]:
from droneFly import aggregate, detect_peak

from pathlib import Path
import pandas as pd
import yaml, os

In [ ]:
os.chdir("..")

In [16]:
run_dir = Path("results/2025-07-21/25-07-21_17-28-34")


In [17]:
data = pd.read_csv(run_dir / "drone_state.csv")
with open(run_dir / "experiment_config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [41]:
aggregator=getattr(aggregate, config["agg_cls"])(**config["agg_kwargs"])
detector=getattr(detect_peak, config["pk_cls"])(**config["pk_kwargs"])
metric = config["agg_kwargs"]["metrics"]

In [40]:
data.shape[0]

373

In [42]:
rows = []
n = 400
for row in data.itertuples():
    rows.append(row)
    if row.Index >= n:
        break

In [43]:
rows

[Pandas(Index=0, time_elapsed=0.0, pitch=0, roll=0, yaw=134, vgx=0, vgy=0, vgz=0, templ=86, temph=89, tof=6553, h=0, bat=56, baro=-1.65, time=0, agx=10.0, agy=-5.0, agz=-999.0),
 Pandas(Index=1, time_elapsed=0.055, pitch=0, roll=0, yaw=134, vgx=0, vgy=0, vgz=0, templ=86, temph=89, tof=6553, h=0, bat=56, baro=-1.65, time=0, agx=10.0, agy=-5.0, agz=-999.0),
 Pandas(Index=2, time_elapsed=0.11, pitch=0, roll=0, yaw=134, vgx=0, vgy=0, vgz=0, templ=86, temph=89, tof=6553, h=0, bat=56, baro=-1.61, time=0, agx=7.0, agy=-7.0, agz=-1000.0),
 Pandas(Index=3, time_elapsed=0.165, pitch=0, roll=0, yaw=134, vgx=0, vgy=0, vgz=0, templ=86, temph=89, tof=6553, h=0, bat=56, baro=-1.61, time=0, agx=7.0, agy=-7.0, agz=-1000.0),
 Pandas(Index=4, time_elapsed=0.22, pitch=0, roll=0, yaw=134, vgx=0, vgy=0, vgz=0, templ=86, temph=89, tof=6553, h=0, bat=56, baro=-1.62, time=0, agx=8.0, agy=-6.0, agz=-999.0),
 Pandas(Index=5, time_elapsed=0.272, pitch=0, roll=0, yaw=134, vgx=0, vgy=0, vgz=0, templ=86, temph=89, t

In [44]:
for row in rows:
    # raw_values = [*map(lambda x: getattr(rows[0], x), metric)]
    agg_value = aggregator(row)
    bump = detector(agg_value)
    print(f"Row {row.Index}, agx: {row.agx}, agy: {row.agy}, agz: {row.agz}, agg_value: {agg_value}, bump: {bump}")

Row 0, agx: 10.0, agy: -5.0, agz: -999.0, agg_value: 0.0, bump: False
Row 1, agx: 10.0, agy: -5.0, agz: -999.0, agg_value: 0.0, bump: False
Row 2, agx: 7.0, agy: -7.0, agz: -1000.0, agg_value: 6.0, bump: False
Row 3, agx: 7.0, agy: -7.0, agz: -1000.0, agg_value: 6.0, bump: False
Row 4, agx: 8.0, agy: -6.0, agz: -999.0, agg_value: 9.0, bump: False
Row 5, agx: 8.0, agy: -6.0, agz: -999.0, agg_value: 9.0, bump: False
Row 6, agx: 9.0, agy: -6.0, agz: -1001.0, agg_value: 6.0, bump: False
Row 7, agx: 9.0, agy: -6.0, agz: -1001.0, agg_value: 6.0, bump: False
Row 8, agx: -1.0, agy: -12.0, agz: -998.0, agg_value: 22.0, bump: False
Row 9, agx: -1.0, agy: -12.0, agz: -998.0, agg_value: 22.0, bump: False
Row 10, agx: 7.0, agy: -8.0, agz: -1002.0, agg_value: 35.0, bump: False
Row 11, agx: 7.0, agy: -8.0, agz: -1002.0, agg_value: 35.0, bump: False
Row 12, agx: 9.0, agy: -9.0, agz: -999.0, agg_value: 22.0, bump: False
Row 13, agx: 34.0, agy: -15.0, agz: -990.0, agg_value: 62.0, bump: False
Row 14, ag

In [30]:
[*map(lambda x: getattr(rows[0], x), metric)]

[10.0, -5.0, -999.0]

In [35]:
aggregator.memory

deque([[3.0, 4.0, -1085.0],
       [3.0, 4.0, -1085.0],
       [8.0, -5.0, -1172.0],
       [8.0, -5.0, -1172.0],
       [17.0, -7.0, -1316.0]],
      maxlen=5)

In [39]:
aggregate.diff_filter(aggregator.memory)

array([ 14.,  11., 231.])

In [38]:
agg_value

np.float64(256.0)

In [32]:
detector.sample

deque([np.float64(0.0),
       np.float64(6.0),
       np.float64(6.0),
       np.float64(9.0),
       np.float64(9.0),
       np.float64(6.0),
       np.float64(6.0),
       np.float64(22.0),
       np.float64(22.0),
       np.float64(35.0),
       np.float64(35.0),
       np.float64(22.0),
       np.float64(62.0),
       np.float64(46.0),
       np.float64(98.0),
       np.float64(201.0),
       np.float64(161.0),
       np.float64(262.0),
       np.float64(210.0),
       np.float64(256.0)],
      maxlen=20)

In [33]:
detector.mean, detector.std, detector.threshold

(np.float64(73.7), np.float64(88.244036625712), 10)

In [34]:
detector.calculate(agg_value)

np.float64(2.0658619774300147)